# Spectral Volume Models on Crypto (BTC / ETH / SOL)

复现 Wu, Zhang, Dai (2025, *Management Science*) "Spectral Volume Models" 框架，把它从美股/A 股搬到 7×24 的 perp 现货市场。

## 研究问题

1. **波段结构**：用频谱方法确认 BTC / ETH / SOL 的成交量在哪些 *高频周期* 上有显著的"算法节奏"。
   - 论文在 US 上找到 10s/15s/20s/30s/1min/5min；在 China 上是 30s/1min/2.5min/5min/10min。
   - 经验先验：算法越成熟 → 主导周期越短。所以我希望验证 **BTC 的主导周期最短（最快）**，SOL 最慢。
2. **是否能转成可交易信号**：论文用 cross-sectional `peri` 排序构建 long-short Periodic-minus-Smooth (PmS) 组合。
   - 我们只有 3 个标的，做不了 cross-section。我们改成 **time-series 维度**：当 `peri` 升高（算法占优）时是否伴随特定的方向性收益？
   - 这能告诉我们：周期性是单纯的"成交节奏"，还是带有 alpha 的 microstructure 信号。

## 现有数据 & 路线

- 当前 cache：`SOL/USDC` Binance USDT-M perp aggTrades（2026-01-01 → 2026-05-09，约 129 个 UTC 日，~500 Mio trades 量级）。
- 框架完全 *symbol-agnostic*，BTC/ETH 拉到同样 cache 后直接调用 `analyze_symbol` 即可。

## Notebook 结构

- Part 1 — Fourier 分析直觉（讲清楚 *为什么* 论文要做这三步）
- Part 2 — 数据加载与 3 秒 bin 的 *intraday panel*
- Part 3 — 三步法估计：detrend → ACF → inverse DFT
- Part 4 — 合成数据 sanity check
- Part 5 — SOL 单标的频谱与 fVR
- Part 6 — 跨标的对比模板（BTC/ETH ready-to-plug）
- Part 7 — Daily `peri` 时序与 forward-return 分析
- Part 8 — 结论与下一步

# Part 1 — Fourier 分析 101（搞清楚论文为什么这么做）

## 1.1 一句话直觉

**任何 *足够规则* 的时间序列都可以写成不同周期的余弦/正弦的叠加。**

\[
V_t \;=\; m_t \;+\; \underbrace{\sum_{j=1}^{n} a_j \cos(\lambda_j t)}_{\text{周期成分}} \;+\; \varepsilon_t
\]

- \(m_t\)：缓慢变化的 *趋势*（美股的 U 形 intraday、加密的 UTC 周期都属于这一项）；
- \(a_j \cos(\lambda_j t)\)：第 \(j\) 个 "频率" 的纯周期信号，\(\lambda_j = j\pi/n\)，对应周期 \(P_j = \frac{2n}{j}\) 个 bin；
- \(\varepsilon_t\)：白噪声（IID，期望 0，方差 \(\sigma^2\)）。

**关键问题**：单个标的、单天里 \(a_j\) 是淹没在噪声里的（论文里 fVR ≈ 1%–10% vs 噪声主导）。所以不能"对原始序列做一次 FFT 完事"，必须先**降噪**再估 \(a_j\)。

## 1.2 为什么先 ACF，后 DFT？（这是论文最聪明的一步）

DFT 直接作用在原始 \(V_t\) 上得到的频谱 \(\hat V(\omega)\)，其期望就是真正的频谱密度 \(S(\omega)\)，但**方差不会随 \(T \to \infty\) 收缩**——这就是为什么 raw periodogram 总是看起来"毛刺"的原因（统计学上叫 *inconsistent* estimator）。

而 *自协方差函数* (ACF):
\[
\gamma(h) = \mathbb{E}[(V_t - \mu)(V_{t+h} - \mu)]
\]
有一个非常漂亮的性质——**Wiener–Khinchin 定理**：

\[
\gamma(h) \;=\; \int_0^\pi S(\omega)\,\cos(\omega h)\,d\omega
\]

也就是说，**ACF 和功率谱密度互为 Fourier 变换对**。论文做的事就是：

1. 估计 \(\hat\gamma(h)\)（这一步对每个 \(h\) 是 *样本均值*，能稳定收敛）。
2. 对 \(\hat\gamma\) 做 *逆 DFT*，就直接拿到了 \(\hat a_j^2\)（不需要先估 \(a_j\) 再平方，这又给了 1× 数值稳定性）。

直观上：**raw V_t 里每个频率的信号被 \(O(\sqrt T)\) 噪声压住**；而**对 \(\gamma(h)\) 来说，噪声只贡献 \(\gamma_\varepsilon(0) = \sigma^2\) 的那一根尖钉**（在 \(h=0\) 处），其它 lag \(h \neq 0\) 的噪声期望为 0——周期信号在 \(\gamma(h)\) 里 *几乎不被噪声污染*。

> **类比**：你听不清 100 米外朋友喊的"你-好-吗"（一个三音节的周期），因为风太大；但如果朋友 *持续重复* 喊一万次然后你做一个"自身延迟 0.5 秒后听到自己回声有多像"的实验，回声里清楚地有一个 0.5 秒周期——这就是 ACF 干的事。

## 1.3 三步法的数学全貌

第 \(j\) 个频率的强度 \(a_j^2\) 的估计：
\[
\hat a_j^2 \;=\; \frac{2}{n}\!\left( 2\sum_{h=1}^{n} \hat\gamma(h)\,\cos\!\Big(\frac{j\pi h}{n}\Big) + \hat\gamma(0) \right)
\]

它来自一个简单的恒等式（论文 Theorem 1）：在 \(T \to \infty\) 下，
\[
\hat\gamma(h) \;\to\; \frac{1}{2}\sum_{j=1}^{n} a_j^2 \cos\!\Big(\frac{j\pi h}{n}\Big) \;+\; \mathbb{1}_{h=0}\,\sigma^2
\]
也就是说 \(\hat\gamma(h)\) 是 \((a_n^2/2, \dots, a_1^2/2,\, \sigma^2,\, a_1^2/2, \dots, a_n^2/2)\) 这条对称序列的 DFT。所以反过来对 \(\hat\gamma\) 做 *逆 DFT* 就把 \(a_j^2\) 直接抽出来了。

## 1.4 fVR：每个频率"解释了多少方差"

由 \(\hat\gamma(0) = \sigma^2 + \tfrac{1}{2}\sum a_j^2\)，所以 *去趋势序列* 的总方差就是 \(\hat\gamma(0)\)。论文定义：

\[
\text{fVR}_j \;=\; \frac{a_j^2}{2\sigma^2 + \sum_i a_i^2} \;=\; \frac{a_j^2}{2\hat\gamma(0)}
\]

含义：**第 \(j\) 个周期解释了去趋势成交量方差的百分之几**。
- 没有任何周期性时（纯白噪声），\(\sum_i a_i^2 = 0\)，但因为我们放了 \(n=500\) 个基函数，每个 \(a_j^2\) 估计值会均匀贴近 \(\sigma^2/n\)，于是 baseline fVR \(= 1/n = 0.2\%\)。
- 论文实证 fVR \(\approx 2\%\sim 10\%\)，所以是 baseline 的 10–50 倍——这就是"显著周期"。

## 1.5 加密市场的差别

| 维度 | US/CN 股票 | Crypto perp |
|---|---|---|
| 交易时段 | 6.5h / 4h | 24h × 7 |
| 截止日历日 | 自然日 | UTC 日 |
| Trend \(m_t\) | U 形 / 双 U | 平台期 + 弱日内（伦敦/纽约盘活跃度高） |
| 数据频率 | tick → 3s | aggTrade → 3s（直接 resample） |

**3 秒 bin / \(n=500\) / \(q=100\)** 的设定可以原样照搬：
- 一个 UTC 日 = `86400 / 3 = 28800` 个 bin；
- detrend 用 \(2q+1 = 201\) bin 的滑动均值（≈ 10 分钟）；
- 最长可探测周期 = \(2n \times 3\text{s} = 50\) 分钟，最短 \(\approx 6\) 秒。

# Part 2 — 数据加载

数据约定：
- Cache 在 `/Volumes/Lexar/mean_reversion_data/cache/{SYMBOL_DIR}/trades_perp/{YYYY-MM-DD}.parquet`，由 `scripts/fetch_perpetual_trades.py` 生成。
- 列：`timestamp, price, volume, side, value, agg_trade_id, first_trade_id, last_trade_id, n_trades_in_agg`。
- 当前可用：`SOL/USDC` 2026-01-01 → 2026-05-09。BTC/ETH 留给后续 fetch。

下面的 `load_perp_trades` 是单文件级的薄封装（直接复用 `rule_backtest.data_loader.load_cached_trades`），让我们能：
- 限制日期范围；
- 切换 symbol（BTC/ETH 准备好就直接 plug in）；
- 在数据缺失时优雅返回，不阻塞 notebook 其它部分。

In [ ]:
from __future__ import annotations

import sys
import logging
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve


def _locate_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for c in [cwd, *cwd.parents]:
        if (c / "rule_backtest").exists() and (c / "utilities").exists():
            return c
    raise FileNotFoundError("Cannot locate repo root containing rule_backtest and utilities")


ROOT = _locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rule_backtest.data_loader import load_cached_trades  # noqa: E402

CACHE_ROOT = Path("/Volumes/Lexar/mean_reversion_data/cache")
CACHE_SUBDIR = "trades_perp"

PRIMARY_SYMBOL = "SOL/USDC"
COMPARISON_SYMBOLS = ["BTC/USDC", "ETH/USDC", "SOL/USDC"]

BIN_SECONDS = 3
N_BASIS = 500
DETREND_HALFWIN = 100
SECONDS_PER_DAY = 24 * 3600
BINS_PER_DAY = SECONDS_PER_DAY // BIN_SECONDS

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("spectral_volume")
logger.setLevel(logging.INFO)

print(f"Repo root        : {ROOT}")
print(f"Cache root       : {CACHE_ROOT}")
print(f"Bin seconds      : {BIN_SECONDS}")
print(f"Basis frequencies: n = {N_BASIS}")
print(f"Detrend window   : 2q+1 = {2 * DETREND_HALFWIN + 1} bins ({(2 * DETREND_HALFWIN + 1) * BIN_SECONDS} sec)")
print(f"Period range     : {2 * BIN_SECONDS}s — {2 * N_BASIS * BIN_SECONDS}s "
      f"({2 * BIN_SECONDS}s — {2 * N_BASIS * BIN_SECONDS / 60:.1f} min)")

In [ ]:
def list_cached_days(symbol: str, cache_root: Path = CACHE_ROOT,
                     cache_subdir: str = CACHE_SUBDIR) -> List[pd.Timestamp]:
    """Return sorted list of UTC dates available in cache for a symbol."""
    sym_dir = cache_root / symbol.replace("/", "_") / cache_subdir
    if not sym_dir.exists():
        return []
    files = sorted(f for f in sym_dir.glob("*.parquet") if not f.name.startswith("._"))
    return [pd.Timestamp(f.stem, tz="UTC") for f in files]


def load_perp_trades(
    symbol: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    cache_root: Path = CACHE_ROOT,
    cache_subdir: str = CACHE_SUBDIR,
) -> pd.DataFrame:
    """Load aggTrades from local cache, optionally restricted by [start, end] (inclusive)."""
    sym_dir = cache_root / symbol.replace("/", "_") / cache_subdir
    if not sym_dir.exists():
        raise FileNotFoundError(f"Cache directory not found: {sym_dir}")

    files = sorted(f for f in sym_dir.glob("*.parquet") if not f.name.startswith("._"))
    if start_date is not None:
        files = [f for f in files if f.stem >= start_date]
    if end_date is not None:
        files = [f for f in files if f.stem <= end_date]
    if not files:
        raise FileNotFoundError(f"No parquet files in {sym_dir} for [{start_date}, {end_date}]")

    dfs = [pd.read_parquet(f) for f in files]
    trades = pd.concat(dfs, ignore_index=True)
    trades["timestamp"] = pd.to_datetime(trades["timestamp"], utc=True)
    trades.sort_values("timestamp", inplace=True)
    trades.reset_index(drop=True, inplace=True)
    trades.rename(columns={"amount": "volume", "cost": "value"}, inplace=True)
    return trades


print("Available days per symbol in cache:")
for sym in COMPARISON_SYMBOLS:
    days = list_cached_days(sym)
    if days:
        print(f"  {sym:10s}: {len(days)} days, {days[0].date()} → {days[-1].date()}")
    else:
        print(f"  {sym:10s}: (no cache yet — fetch via scripts/fetch_perpetual_trades.py)")

## 2.1 构建 *intraday panel* — `(day, second_of_day) → measure`

我们要复现论文里 Figure 1(c)(d) 的 *cross-day average* 曲线。流程：

1. 把每条 trade 落到它所属的 *3 秒 bin*，bin 的 key = `(UTC date, second_of_day // 3)`。
2. 在每个 `(day, bin)` 上统计三种 volume 度量（论文也是这三种）：
   - `n_trades`：该 bin 内的 aggTrade 笔数（**主要**信号，最纯的算法节奏）；
   - `volume`：成交数量（基础币）；
   - `value`：成交金额（USDC）。
3. 把它 pivot 成 `(D × B)` 的矩阵，缺失日/缺失 bin 填 0。
4. 横向求每个 bin 跨日均值，即得到论文里的 \(\bar V_t\)。

> 论文 §5.1.1 强调：**`n_trades` 的 fVR 比 `volume` 和 `value` 都强**——因为算法把大单切碎，节奏体现在"次数"而不是"体量"上。我们后面会用 `n_trades` 当主信号，另外两种作为 robustness 检查。

In [ ]:
@dataclass
class IntradayPanel:
    """3-second intraday volume panel keyed by (UTC day, bin)."""

    symbol: str
    bin_seconds: int
    days: pd.DatetimeIndex
    bin_edges: np.ndarray
    n_trades: np.ndarray
    volume: np.ndarray
    value: np.ndarray

    @property
    def n_days(self) -> int:
        return len(self.days)

    @property
    def n_bins(self) -> int:
        return len(self.bin_edges)

    def avg(self, measure: str = "n_trades") -> np.ndarray:
        """Cross-day mean of the chosen measure (论文里的 V_{t,s,·})."""
        arr = getattr(self, measure)
        return arr.mean(axis=0)

    def daily(self, measure: str = "n_trades") -> np.ndarray:
        """(D × B) matrix for per-day spectral analysis."""
        return getattr(self, measure)


def build_intraday_panel(
    trades: pd.DataFrame,
    symbol: str = "?",
    bin_seconds: int = BIN_SECONDS,
    drop_partial_days: bool = True,
) -> IntradayPanel:
    """Bin aggTrades into a (day × second_of_day) panel for n_trades / volume / value."""
    if trades.empty:
        raise ValueError("trades is empty")

    bins_per_day = SECONDS_PER_DAY // bin_seconds
    ts = trades["timestamp"]
    sec_of_day = (ts.dt.hour * 3600 + ts.dt.minute * 60 + ts.dt.second).to_numpy(np.int64)
    bin_idx = sec_of_day // bin_seconds
    day = ts.dt.normalize()

    df = pd.DataFrame({
        "day": day.values,
        "bin": bin_idx,
        "n_trades": np.ones(len(trades), dtype=np.int64),
        "volume": trades["volume"].to_numpy(np.float64),
        "value": (
            trades["value"].to_numpy(np.float64) if "value" in trades.columns
            else (trades["price"].to_numpy(np.float64) * trades["volume"].to_numpy(np.float64))
        ),
    })
    grouped = df.groupby(["day", "bin"], sort=True).agg(
        n_trades=("n_trades", "sum"),
        volume=("volume", "sum"),
        value=("value", "sum"),
    ).reset_index()

    days_all = pd.DatetimeIndex(sorted(grouped["day"].unique()), name="day")
    bins_all = np.arange(bins_per_day, dtype=np.int64)

    if drop_partial_days:
        partial = []
        for d, sub in grouped.groupby("day"):
            covered = sub["bin"].max() - sub["bin"].min() + 1
            if covered < 0.95 * bins_per_day:
                partial.append(d)
        if partial:
            grouped = grouped[~grouped["day"].isin(partial)]
            days_all = pd.DatetimeIndex(sorted(grouped["day"].unique()), name="day")
            print(f"  dropped {len(partial)} partial day(s): {[str(d.date()) for d in partial]}")

    def _pivot(col: str) -> np.ndarray:
        out = (
            grouped.pivot(index="day", columns="bin", values=col)
            .reindex(index=days_all, columns=bins_all)
            .fillna(0.0)
            .to_numpy(np.float64)
        )
        return out

    return IntradayPanel(
        symbol=symbol,
        bin_seconds=bin_seconds,
        days=days_all,
        bin_edges=bins_all * bin_seconds,
        n_trades=_pivot("n_trades"),
        volume=_pivot("volume"),
        value=_pivot("value"),
    )


print("Loading SOL/USDC perp trades...")
sol_trades = load_perp_trades(PRIMARY_SYMBOL)
print(f"  trades   : {len(sol_trades):,}")
print(f"  span     : {sol_trades['timestamp'].iloc[0]} → {sol_trades['timestamp'].iloc[-1]}")

print("Building intraday panel...")
panel_sol = build_intraday_panel(sol_trades, symbol=PRIMARY_SYMBOL)
print(f"  shape    : {panel_sol.n_days} days × {panel_sol.n_bins} bins ({panel_sol.bin_seconds}s each)")
print(f"  span     : {panel_sol.days[0].date()} → {panel_sol.days[-1].date()}")
print(f"  per-day mean trades : {panel_sol.n_trades.sum(axis=1).mean():,.0f}")
print(f"  per-bin mean trades : {panel_sol.n_trades.mean():.3f}")

In [ ]:
def plot_intraday_average(panel: IntradayPanel, measure: str = "n_trades",
                          ax: Optional[plt.Axes] = None, color: str = "C0") -> plt.Axes:
    """Reproduces论文 Figure 1(c)(d): cross-day average of intraday volume."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(13, 3.5))
    avg = panel.avg(measure)
    hours = panel.bin_edges / 3600.0
    ax.plot(hours, avg, color=color, lw=0.6, label=f"{panel.symbol} ({measure})")
    for h in range(0, 25):
        ax.axvline(h, color="gray", lw=0.3, alpha=0.3)
    ax.set_xlim(0, 24)
    ax.set_xlabel("Hour of UTC day")
    ax.set_ylabel(f"Avg {measure} per {panel.bin_seconds}s bin")
    ax.set_title(f"{panel.symbol} — cross-day average intraday {measure} "
                 f"({panel.n_days} days)")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
    return ax


fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
plot_intraday_average(panel_sol, "n_trades", ax=axes[0], color="C0")
plot_intraday_average(panel_sol, "volume", ax=axes[1], color="C1")
plot_intraday_average(panel_sol, "value", ax=axes[2], color="C2")
plt.tight_layout()
plt.show()

# Part 3 — 三步法估计：detrend → ACF → inverse DFT

下面这个函数把论文 §3.2 的三步严格按公式实现：

```
Step 1  m̂_t = (1 / (2q+1)) Σ_{k=-q..q} V_{t+k}            （滑动均值去趋势）
Step 2  γ̂_X(h) = (1 / (T-2q-h)) Σ_t (X_t - X̄)(X_{t+h} - X̄)   （样本自协方差）
Step 3  â_j² = (2/n) [ 2 Σ_{h=1..n} γ̂(h) cos(jπh/n) + γ̂(0) ]   （逆 DFT）
        fVR_j = â_j² / (2 γ̂(0))                                    （方差占比）
```

实现细节：
- Step 2 用 `scipy.signal.fftconvolve` 跑 \(O(T\log T)\) 而不是 \(O(T n)\)，对长序列（28800+ bin）显著更快；
- Step 3 写成矩阵乘法 `cos_mat @ γ`，便于一次性算出全部 \(j\)；
- 估计出来的 `a²_j` 偶尔会因为有限样本误差变成小负数 — 论文里也会出现（对应 fVR ≈ 0 的频率）。我们如实保留。

In [ ]:
def _rolling_mean_centered(v: np.ndarray, q: int) -> np.ndarray:
    """Rolling mean with window 2q+1, returning the *valid* interior of length T-2q."""
    if 2 * q + 1 > len(v):
        raise ValueError(f"window 2q+1={2*q+1} larger than series len {len(v)}")
    kernel = np.ones(2 * q + 1, dtype=np.float64) / (2 * q + 1)
    full = np.convolve(v, kernel, mode="valid")
    return full


def _autocov_via_fft(x: np.ndarray, n_lags: int) -> np.ndarray:
    """Unbiased sample autocovariance γ̂(0..n_lags) via FFT (matches paper §3.2).

    γ̂(h) = (1 / (T-h)) Σ_t (x_t - x̄)(x_{t+h} - x̄)
    """
    x = x - x.mean()
    T = len(x)
    full = fftconvolve(x, x[::-1], mode="full")
    raw = full[T - 1: T - 1 + n_lags + 1]
    norm = T - np.arange(n_lags + 1, dtype=np.float64)
    return raw / norm


@dataclass
class SpectralResult:
    symbol: str
    measure: str
    bin_seconds: int
    n: int
    q: int
    j: np.ndarray
    period_seconds: np.ndarray
    a2: np.ndarray
    fVR: np.ndarray
    gamma: np.ndarray
    gamma0: float
    detrended_var: float
    detrended_mean: float
    n_obs: int

    def fVR_at_period(self, period_seconds: float) -> Tuple[int, float, float, float]:
        """Find the *j* whose period is closest to `period_seconds` and report (j, period, a², fVR)."""
        idx = int(np.argmin(np.abs(self.period_seconds - period_seconds)))
        return (
            int(self.j[idx]),
            float(self.period_seconds[idx]),
            float(self.a2[idx]),
            float(self.fVR[idx]),
        )

    def top_k_periods(self, k: int = 6) -> pd.DataFrame:
        """Return top-k frequencies by fVR."""
        order = np.argsort(self.a2)[::-1][:k]
        return pd.DataFrame({
            "j": self.j[order].astype(int),
            "period_sec": self.period_seconds[order].round(2),
            "a2": self.a2[order],
            "fVR": self.fVR[order],
        }).reset_index(drop=True)


def estimate_spectral(
    series: np.ndarray,
    *,
    n: int = N_BASIS,
    q: int = DETREND_HALFWIN,
    bin_seconds: int = BIN_SECONDS,
    symbol: str = "?",
    measure: str = "n_trades",
) -> SpectralResult:
    """Three-step estimation of squared intensities a_j² and frequency variance ratios."""
    series = np.asarray(series, dtype=np.float64)
    if len(series) < 2 * q + 2 * n + 1:
        raise ValueError(
            f"series length {len(series)} < 2q+2n+1 = {2*q + 2*n + 1}; "
            "increase data or reduce n/q"
        )

    m_hat = _rolling_mean_centered(series, q)
    X = series[q: len(series) - q] - m_hat
    detrended_var = float(X.var(ddof=0))
    detrended_mean = float(X.mean())

    gamma = _autocov_via_fft(X, n_lags=n)
    gamma0 = float(gamma[0])

    j = np.arange(1, n + 1, dtype=np.int64)
    h = np.arange(1, n + 1, dtype=np.int64)
    cos_mat = np.cos(np.outer(j, h) * (np.pi / n))
    a2 = (2.0 / n) * (2.0 * cos_mat @ gamma[1:] + gamma[0])
    fVR = a2 / (2.0 * gamma0) if gamma0 > 0 else np.full_like(a2, np.nan)

    period_seconds = (2.0 * n / j) * bin_seconds

    return SpectralResult(
        symbol=symbol,
        measure=measure,
        bin_seconds=bin_seconds,
        n=n,
        q=q,
        j=j,
        period_seconds=period_seconds,
        a2=a2,
        fVR=fVR,
        gamma=gamma,
        gamma0=gamma0,
        detrended_var=detrended_var,
        detrended_mean=detrended_mean,
        n_obs=len(X),
    )


print("Spectral framework loaded.")

# Part 4 — 合成数据 sanity check

为了让你相信这个框架不是黑盒，我们造一个 *已知答案* 的序列：
- 趋势 \(m_t = 5 + 4\sin(2\pi t / T)\)（模拟 U 形 / 日内活跃度变化）；
- 两个真实周期：60 秒（\(a_{\text{60s}} = 1.0\)）和 5 分钟（\(a_{\text{5min}} = 0.7\)）；
- 白噪声 \(\varepsilon_t \sim \mathcal{N}(0, 1.5^2)\)（噪声方差 ≈ 信号方差的 5 倍，模拟 *low SNR*）。

期望框架：
1. \(a^2_j\) 在 60s 和 5min 处出现明显尖峰；
2. 估计出来的 \(a_j\) ≈ 真值 ±10%；
3. 其他频率的 fVR 都接近 baseline \(1/n = 0.2\%\)。

In [ ]:
def _synth_series(
    n_bins: int,
    bin_seconds: int,
    truths: Dict[float, float],
    noise_sigma: float = 1.5,
    trend_amp: float = 4.0,
    seed: int = 7,
) -> Tuple[np.ndarray, Dict[float, int]]:
    """Build a synthetic V_t with known periodic content.  truths: {period_seconds: amplitude}."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_bins, dtype=np.float64)
    series = 5.0 + trend_amp * np.sin(2 * np.pi * t / n_bins)
    j_truth: Dict[float, int] = {}
    for period_sec, amp in truths.items():
        # nearest integer j whose period 2n_basis*bin/j matches the desired period
        j_int = int(round(2 * N_BASIS * bin_seconds / period_sec))
        j_truth[period_sec] = j_int
        lam = j_int * np.pi / N_BASIS
        series = series + amp * np.cos(lam * t)
    series = series + rng.normal(0, noise_sigma, size=n_bins)
    return series, j_truth


N_DAYS_SYNTH = 30
T_synth = N_DAYS_SYNTH * BINS_PER_DAY
truths = {60.0: 1.0, 300.0: 0.7}
synth, j_truth = _synth_series(T_synth, BIN_SECONDS, truths)

res_synth = estimate_spectral(synth, symbol="SYNTH", measure="synthetic")
print(f"Synthetic series: T={T_synth}, n_obs after detrend={res_synth.n_obs}")
print()

print(f"{'truth period':>14} | {'truth a':>8} | {'estim a':>8} | {'rel err':>8} | {'fVR':>7}")
print("-" * 60)
for p_sec, amp_true in truths.items():
    j_t = j_truth[p_sec]
    a2_hat = res_synth.a2[j_t - 1]
    fvr_hat = res_synth.fVR[j_t - 1]
    a_hat = np.sqrt(max(a2_hat, 0.0))
    rel = (a_hat - amp_true) / amp_true * 100.0
    print(f"{p_sec:>14.0f} | {amp_true:>8.3f} | {a_hat:>8.3f} | {rel:>+7.1f}% | {fvr_hat:>7.2%}")

baseline_fvr = 1.0 / N_BASIS
median_other_fvr = np.median(np.delete(res_synth.fVR, [j_truth[p] - 1 for p in truths]))
print()
print(f"baseline fVR (1/n)         : {baseline_fvr:.4%}")
print(f"median fVR at non-true j's : {median_other_fvr:.4%}  (should be close to baseline)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(np.arange(2000), synth[:2000], lw=0.5, color="C7")
axes[0].set_xlabel("bin index (3s)")
axes[0].set_ylabel("synthetic V_t")
axes[0].set_title(f"Synthetic series (first 2000 bins ≈ {2000*BIN_SECONDS/60:.0f} min)\n"
                  "60s + 5min cosine + U-trend + N(0, 1.5²)")
axes[0].grid(alpha=0.3)

axes[1].plot(res_synth.j, res_synth.a2, lw=0.7, color="C0", label="estimated â²_j")
for p_sec, j_t in j_truth.items():
    axes[1].axvline(j_t, color="C3", lw=0.6, ls="--", alpha=0.6)
    axes[1].annotate(f"{p_sec:.0f}s\n(j={j_t})", (j_t, res_synth.a2[j_t - 1]),
                     textcoords="offset points", xytext=(6, 4), color="C3")
axes[1].set_xlabel("frequency index j  (period = 2n·Δt / j)")
axes[1].set_ylabel("â²_j")
axes[1].set_title("Recovered intensity coefficients on synthetic data")
axes[1].set_xscale("log")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Part 5 — SOL/USDC 单标的频谱

把上面的框架跑在 SOL 真实数据上。流程：
1. 从 panel 拿到 cross-day average curve \(\bar V_t\)（28800 个 3 秒 bin）；
2. 三步法估计 \(\hat a_j^2\) 与 fVR；
3. 在常见的 round 周期上读 fVR：6s / 10s / 15s / 20s / 30s / 1min / 2min / 5min / 10min；
4. 画 Figure 2 风格的 \(\hat a_j^2\) vs \(j\)。

In [ ]:
CANONICAL_PERIODS_SEC = [6, 10, 15, 20, 30, 60, 120, 300, 600, 1500]


def fVR_table(result: SpectralResult, periods: Iterable[float] = CANONICAL_PERIODS_SEC) -> pd.DataFrame:
    rows = []
    for p in periods:
        j_, p_actual, a2_, fvr_ = result.fVR_at_period(p)
        rows.append({
            "period_request": p,
            "period_actual": round(p_actual, 2),
            "j": j_,
            "a2": a2_,
            "fVR": fvr_,
        })
    df = pd.DataFrame(rows)
    df["fVR_pct"] = (df["fVR"] * 100).round(3)
    df.attrs["symbol"] = result.symbol
    df.attrs["measure"] = result.measure
    df.attrs["baseline_fVR"] = 1.0 / result.n
    return df


sol_avg_n = panel_sol.avg("n_trades")
res_sol_n = estimate_spectral(sol_avg_n, symbol="SOL/USDC", measure="n_trades")

print(f"SOL/USDC n_trades: T={len(sol_avg_n)}, detrended n_obs={res_sol_n.n_obs}")
print(f"  γ̂(0) (detrended variance) = {res_sol_n.gamma0:.4f}")
print()
tab_sol = fVR_table(res_sol_n)
print(f"Baseline fVR (1/n) = {1/N_BASIS:.4%}")
print(f"SOL/USDC fVR table (n_trades, cross-day average across {panel_sol.n_days} days):")
print(tab_sol.to_string(index=False))

In [ ]:
def plot_intensity_spectrum(result: SpectralResult, marks_sec: Iterable[float] = CANONICAL_PERIODS_SEC,
                            ax: Optional[plt.Axes] = None, color: str = "C0") -> plt.Axes:
    """Reproduces 论文 Figure 2 — squared intensities vs frequency index, with period markers."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(13, 4))
    ax.plot(result.j, result.a2, lw=0.6, color=color, alpha=0.8)
    for p_sec in marks_sec:
        j_, p_actual, a2_, fvr_ = result.fVR_at_period(p_sec)
        ax.scatter(j_, a2_, s=35, color="C3", zorder=5)
        label_ = f"{p_sec}s" if p_sec < 60 else f"{int(p_sec/60)}min"
        ax.annotate(f"{label_}\n{fvr_:.1%}", (j_, a2_),
                    textcoords="offset points", xytext=(4, 6), color="C3", fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("frequency index j  (period = 2n·Δt / j; smaller j = longer period)")
    ax.set_ylabel("â²_j")
    ax.set_title(f"{result.symbol} — squared intensity coefficients ({result.measure})")
    ax.grid(alpha=0.3, which="both")
    ax.axhline(0, color="k", lw=0.4, alpha=0.5)
    return ax


fig, ax = plt.subplots(1, 1, figsize=(13, 4.5))
plot_intensity_spectrum(res_sol_n, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
print("Top-10 SOL/USDC frequencies by â²_j (n_trades):")
print(res_sol_n.top_k_periods(k=10).to_string(index=False))

print()
print("Robustness — same analysis on volume and value:")
res_sol_v = estimate_spectral(panel_sol.avg("volume"), symbol="SOL/USDC", measure="volume")
res_sol_d = estimate_spectral(panel_sol.avg("value"), symbol="SOL/USDC", measure="value")

side_by_side = pd.DataFrame({
    "period_sec": CANONICAL_PERIODS_SEC,
    "fVR_n_trades": [res_sol_n.fVR_at_period(p)[3] for p in CANONICAL_PERIODS_SEC],
    "fVR_volume": [res_sol_v.fVR_at_period(p)[3] for p in CANONICAL_PERIODS_SEC],
    "fVR_value": [res_sol_d.fVR_at_period(p)[3] for p in CANONICAL_PERIODS_SEC],
})
side_by_side[["fVR_n_trades", "fVR_volume", "fVR_value"]] *= 100
side_by_side = side_by_side.round(3)
side_by_side.columns = ["period_sec", "fVR_n_trades%", "fVR_volume%", "fVR_value%"]
print(side_by_side.to_string(index=False))

# Part 6 — 跨标的对比框架（BTC / ETH / SOL）

`analyze_symbol(...)` 把 *load → panel → spectral* 一条龙打包。等 BTC 和 ETH 的 cache 拉好之后（`scripts/fetch_perpetual_trades.py --symbols BTC/USDC,ETH/USDC --start-date 2026-01-01`），就直接调一次 `analyze_symbol("BTC/USDC")` 即可，下游绘图代码会自动把它拼到对比图里。

## 我们想要从对比里看到什么

1. **主导周期的 *快慢*。** 论文里美股以 30s/1min 为主，A 股以 1min/5min 为主——美股更"快"。
   **猜想**：BTC perp 的算法 / 做市最成熟，主导周期最短（10–30s 量级）；SOL 还是以 1min / 5min 为主；ETH 介于其间。
2. **fVR 的 *绝对量级*。** 越大说明算法占比越高、节奏越规则。
3. **是否 *主导周期一致*。** 如果三个标的都在 1min 处共振，意味着同一类策略在多市场跑（典型：跨币 VWAP / 套利机器人）。

In [ ]:
@dataclass
class SymbolAnalysis:
    symbol: str
    panel: IntradayPanel
    spectrum_n: SpectralResult
    spectrum_v: SpectralResult
    spectrum_d: SpectralResult


def analyze_symbol(
    symbol: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    cache_root: Path = CACHE_ROOT,
) -> Optional[SymbolAnalysis]:
    """End-to-end analysis for one symbol.  Returns None if cache missing."""
    sym_dir = cache_root / symbol.replace("/", "_") / CACHE_SUBDIR
    if not sym_dir.exists() or not list(sym_dir.glob("*.parquet")):
        print(f"  [skip] {symbol}: no cache at {sym_dir}")
        return None
    print(f"  [load] {symbol}...")
    trades = load_perp_trades(symbol, start_date=start_date, end_date=end_date)
    panel = build_intraday_panel(trades, symbol=symbol)
    return SymbolAnalysis(
        symbol=symbol,
        panel=panel,
        spectrum_n=estimate_spectral(panel.avg("n_trades"), symbol=symbol, measure="n_trades"),
        spectrum_v=estimate_spectral(panel.avg("volume"), symbol=symbol, measure="volume"),
        spectrum_d=estimate_spectral(panel.avg("value"), symbol=symbol, measure="value"),
    )


print("Running analyze_symbol() for BTC, ETH, SOL (skips missing caches)...")
analyses: Dict[str, SymbolAnalysis] = {}
for sym in COMPARISON_SYMBOLS:
    a = analyze_symbol(sym)
    if a is not None:
        analyses[sym] = a

print()
print(f"Symbols completed: {list(analyses)}")

In [ ]:
def cross_symbol_fVR_table(analyses: Dict[str, SymbolAnalysis],
                           periods: Iterable[float] = CANONICAL_PERIODS_SEC,
                           measure: str = "n_trades") -> pd.DataFrame:
    """Side-by-side fVR table across symbols (one column per symbol)."""
    rows = []
    for p in periods:
        row = {"period_sec": p}
        for sym, a in analyses.items():
            res = getattr(a, {"n_trades": "spectrum_n", "volume": "spectrum_v",
                              "value": "spectrum_d"}[measure])
            row[sym] = res.fVR_at_period(p)[3] * 100  # in %
        rows.append(row)
    return pd.DataFrame(rows).round(3)


def plot_cross_symbol_intensity(analyses: Dict[str, SymbolAnalysis], measure: str = "n_trades"):
    if not analyses:
        print("No symbols loaded — fetch BTC/ETH cache first.")
        return
    fig, ax = plt.subplots(1, 1, figsize=(13, 4.5))
    palette = {"BTC/USDC": "C3", "ETH/USDC": "C2", "SOL/USDC": "C0"}
    for sym, a in analyses.items():
        res = getattr(a, {"n_trades": "spectrum_n", "volume": "spectrum_v",
                          "value": "spectrum_d"}[measure])
        ax.plot(res.j, res.a2 / res.a2.max(), lw=0.8, alpha=0.85,
                color=palette.get(sym, None), label=f"{sym}  ({a.panel.n_days}d)")
    for p_sec in [10, 30, 60, 300, 600]:
        j_ = int(round(2 * N_BASIS * BIN_SECONDS / p_sec))
        label_ = f"{p_sec}s" if p_sec < 60 else f"{int(p_sec/60)}min"
        ax.axvline(j_, color="gray", lw=0.4, alpha=0.5)
        ax.text(j_, 1.02, label_, color="gray", fontsize=8, ha="center", va="bottom")
    ax.set_xscale("log")
    ax.set_xlabel("frequency index j  (period = 2n·Δt / j)")
    ax.set_ylabel("â²_j  (max-normalised per symbol)")
    ax.set_title(f"Cross-symbol intensity spectra ({measure})  — peaks closer to right ⇒ faster algorithms")
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


print("Cross-symbol fVR table (n_trades, %):")
print(cross_symbol_fVR_table(analyses, measure="n_trades").to_string(index=False))

plot_cross_symbol_intensity(analyses, measure="n_trades")

In [ ]:
def dominant_period_summary(analyses: Dict[str, SymbolAnalysis], measure: str = "n_trades",
                            top_k: int = 3) -> pd.DataFrame:
    """For each symbol report top-K periods (in seconds) by fVR — main 'speed' diagnostic."""
    rows = []
    for sym, a in analyses.items():
        res = getattr(a, {"n_trades": "spectrum_n", "volume": "spectrum_v",
                          "value": "spectrum_d"}[measure])
        top = res.top_k_periods(k=top_k).copy()
        top["symbol"] = sym
        top["rank"] = np.arange(1, len(top) + 1)
        rows.append(top)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    return out[["symbol", "rank", "j", "period_sec", "a2", "fVR"]]


print(f"\nTop-3 周期排序 (per symbol, by â²_j on n_trades) — speed diagnostic:")
print(dominant_period_summary(analyses, top_k=3).to_string(index=False))

# Part 7 — Daily `peri` 时序与 forward-return

## 7.1 Daily `peri` 的定义

论文的核心信号是 *每天每只票的 `peri`*：

\[
\text{peri}_{s,d} \;=\; \sum_{p \in P^\star} \text{fVR}_{p,\,s,\,d}
\]

其中 \(P^\star\) 是市场层面的"主导周期集合"。论文里：
- US 是 {10s, 15s, 20s, 30s, 1min, 5min}；
- CN 是 {30s, 1min, 2.5min, 5min, 10min}。

**含义**：peri 越高 → 当天该标的 *算法节奏越强* → 算法对成交节奏的占比越高。

我们用 SOL Part 5 的结果去定义它的"主导周期集合" `P_sol_star`，BTC / ETH 一旦数据进来再各自定义。

## 7.2 论文的 PmS 信号 vs 我们的版本

| 维度 | 论文 PmS | 我们的版本 |
|---|---|---|
| 截面宽度 | 500–2000 只票 | 3 只 perp |
| 每月排序 | top vs bottom 五分位 | 不可行 |
| 替代 | — | **time-series**：把单标的的 daily peri 分位，比较高 peri 日 vs 低 peri 日的 *next-day return* |

逻辑：如果"算法占比强 → 信息不对称强 → 投资者要求 risk premium"在加密上也成立，那么 *peri 高的日子* 之后的 forward-return 应当（在均值层面）显著高于 peri 低的日子。这个能复制论文 §6.3 在单标的上的退化版本。

In [ ]:
def dominant_periods_set(result: SpectralResult, top_k: int = 5,
                         min_period_sec: float = 6.0) -> List[float]:
    """Pick the symbol's own dominant period set P*: top-k periods by â² with period ≥ min."""
    df = result.top_k_periods(k=N_BASIS)
    df = df[df["period_sec"] >= min_period_sec].head(top_k)
    return df["period_sec"].tolist()


def daily_peri_series(panel: IntradayPanel, period_set: Iterable[float],
                      measure: str = "n_trades", n: int = N_BASIS,
                      q: int = DETREND_HALFWIN) -> pd.DataFrame:
    """Compute daily peri_d = Σ_{p ∈ P*} fVR_{p,d}, one row per UTC day.

    For each day we run the spectral estimation on *that day's* intraday curve.
    """
    daily = panel.daily(measure)
    rows = []
    for i, day in enumerate(panel.days):
        try:
            res_d = estimate_spectral(
                daily[i], n=n, q=q, bin_seconds=panel.bin_seconds,
                symbol=panel.symbol, measure=measure,
            )
        except ValueError:
            continue
        fvr_sum = 0.0
        per_period = {}
        for p in period_set:
            j_, p_actual, _, fvr_ = res_d.fVR_at_period(p)
            per_period[f"fVR_{int(round(p_actual))}s"] = fvr_
            fvr_sum += fvr_
        rows.append({
            "day": day,
            "peri": fvr_sum,
            "gamma0": res_d.gamma0,
            "n_trades_total": float(daily[i].sum()),
            **per_period,
        })
    out = pd.DataFrame(rows).set_index("day")
    return out.sort_index()


P_sol_star = dominant_periods_set(res_sol_n, top_k=5, min_period_sec=6.0)
print(f"SOL dominant period set P* (top-5 by â² from cross-day spectrum): {P_sol_star} sec")

print("Computing daily peri series for SOL...")
peri_sol = daily_peri_series(panel_sol, P_sol_star)
print(f"  produced {len(peri_sol)} day(s) of peri values")
print(peri_sol.head())
print()
print(peri_sol[["peri"]].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)

axes[0].plot(peri_sol.index, peri_sol["peri"], color="C0", lw=1.0)
axes[0].set_ylabel("daily peri\n(Σ fVR over P*)")
axes[0].set_title(f"{PRIMARY_SYMBOL} — daily peri (sum of fVR at {P_sol_star} sec)")
axes[0].grid(alpha=0.3)
axes[0].axhline(peri_sol["peri"].mean(), color="C3", lw=0.6, ls="--",
                label=f"mean={peri_sol['peri'].mean():.3f}")
axes[0].legend()

for col in [c for c in peri_sol.columns if c.startswith("fVR_")]:
    axes[1].plot(peri_sol.index, peri_sol[col], lw=0.8, alpha=0.85, label=col)
axes[1].set_ylabel("per-period fVR")
axes[1].set_xlabel("UTC day")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper left", ncol=3, fontsize=8)

plt.tight_layout()
plt.show()

## 7.3 peri 与 forward return 的关系

简单 event study：
- 用 `peri_d` 的滚动 60 日分位（避免 lookahead），把每天打成 `peri_quintile_d ∈ {1, 2, 3, 4, 5}`；
- 计算 1 / 3 / 5 / 10 day 的 forward log return；
- 看 quintile 5 (peri 最高) - quintile 1 (peri 最低) 的 mean / median forward return 差异；
- 同步报告 t-stat（Newey-West HAC，简化为 standard error w/ lag adjustment）。

**期望**：如果论文逻辑迁移得过来，peri-high 日之后的 forward return 应当更高（信息不对称的 risk premium）。**警示**：单标的 + 100+ 天的小样本下，t-stat 大概率不显著；这部分主要是把 *分析框架* 准备好。

In [ ]:
def compute_daily_close(trades: pd.DataFrame) -> pd.Series:
    """Per-UTC-day close price (last trade)."""
    df = trades[["timestamp", "price"]].copy()
    df["day"] = df["timestamp"].dt.normalize()
    return df.groupby("day")["price"].last().rename("close")


def forward_log_returns(close: pd.Series, horizons: Iterable[int] = (1, 3, 5, 10)) -> pd.DataFrame:
    log_close = np.log(close)
    out = {}
    for h in horizons:
        out[f"ret_fwd_{h}d"] = (log_close.shift(-h) - log_close)
    return pd.DataFrame(out, index=close.index)


def peri_quintile(peri_series: pd.Series, lookback_days: int = 60, n_q: int = 5) -> pd.Series:
    """Rolling expanding-then-rolling quantile rank to avoid lookahead."""
    ranks = pd.Series(np.nan, index=peri_series.index)
    for i in range(len(peri_series)):
        if i < lookback_days:
            continue
        window = peri_series.iloc[i - lookback_days: i + 1].dropna()
        if len(window) < n_q * 2:
            continue
        cur = peri_series.iloc[i]
        # rank in window, scaled to 1..n_q
        pct = (window <= cur).mean()
        q = int(np.clip(np.ceil(pct * n_q), 1, n_q))
        ranks.iloc[i] = q
    return ranks


close_sol = compute_daily_close(sol_trades)
fwd_sol = forward_log_returns(close_sol, horizons=(1, 3, 5, 10))

peri_eval = peri_sol[["peri"]].copy()
peri_eval["close"] = close_sol.reindex(peri_eval.index)
peri_eval = peri_eval.join(fwd_sol)
peri_eval["quintile"] = peri_quintile(peri_eval["peri"], lookback_days=60)
peri_eval = peri_eval.dropna(subset=["quintile"])
print(f"Eval window after warm-up: {len(peri_eval)} days  ({peri_eval.index.min().date()} → {peri_eval.index.max().date()})")
print()
print("Forward-return mean by peri quintile (1=low peri, 5=high peri):")
agg = peri_eval.groupby("quintile")[["ret_fwd_1d", "ret_fwd_3d", "ret_fwd_5d", "ret_fwd_10d"]].agg(
    ["mean", "count"]
)
print((agg * 100).round(3) if False else agg.round(4).to_string())

In [ ]:
def quintile_long_short(eval_df: pd.DataFrame,
                        ret_cols: Iterable[str] = ("ret_fwd_1d", "ret_fwd_3d", "ret_fwd_5d", "ret_fwd_10d"),
                        n_q: int = 5) -> pd.DataFrame:
    """Mean and t-stat for the (Q5 - Q1) long-short on each forward horizon."""
    rows = []
    for col in ret_cols:
        q5 = eval_df.loc[eval_df["quintile"] == n_q, col].dropna()
        q1 = eval_df.loc[eval_df["quintile"] == 1, col].dropna()
        if len(q5) < 3 or len(q1) < 3:
            rows.append({"horizon": col, "n_q5": len(q5), "n_q1": len(q1),
                         "mean_q5": np.nan, "mean_q1": np.nan,
                         "diff": np.nan, "t_stat": np.nan})
            continue
        mu5, mu1 = q5.mean(), q1.mean()
        diff = mu5 - mu1
        s5, s1 = q5.std(ddof=1), q1.std(ddof=1)
        n5, n1 = len(q5), len(q1)
        se = float(np.sqrt(s5**2 / n5 + s1**2 / n1))
        t_stat = diff / se if se > 0 else np.nan
        rows.append({"horizon": col, "n_q5": n5, "n_q1": n1,
                     "mean_q5": mu5, "mean_q1": mu1, "diff": diff, "t_stat": t_stat})
    return pd.DataFrame(rows)


ls_table = quintile_long_short(peri_eval)
ls_table_disp = ls_table.copy()
for c in ["mean_q5", "mean_q1", "diff"]:
    ls_table_disp[c] = (ls_table_disp[c] * 100).round(3).astype(str) + "%"
ls_table_disp["t_stat"] = ls_table_disp["t_stat"].round(2)
print(f"\nQ5(high peri) − Q1(low peri) forward log-return spread, {PRIMARY_SYMBOL}:")
print(ls_table_disp.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

q_means = peri_eval.groupby("quintile")[["ret_fwd_1d", "ret_fwd_5d", "ret_fwd_10d"]].mean()
for col in q_means.columns:
    axes[0].plot(q_means.index, q_means[col] * 100, marker="o", label=col)
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_xlabel("peri quintile (1=low, 5=high)")
axes[0].set_ylabel("mean forward log-return (%)")
axes[0].set_title(f"{PRIMARY_SYMBOL} — forward return by peri quintile")
axes[0].grid(alpha=0.3)
axes[0].legend()

eval_sorted = peri_eval.sort_values("peri")
axes[1].scatter(eval_sorted["peri"], eval_sorted["ret_fwd_5d"] * 100, s=14, alpha=0.6, color="C0")
m, b = np.polyfit(eval_sorted["peri"].values, (eval_sorted["ret_fwd_5d"] * 100).values, 1)
xs = np.linspace(eval_sorted["peri"].min(), eval_sorted["peri"].max(), 50)
axes[1].plot(xs, m * xs + b, color="C3", lw=1.0,
             label=f"OLS: {m:+.3f}·peri + {b:+.3f}")
axes[1].set_xlabel("peri")
axes[1].set_ylabel("5-day forward log-return (%)")
axes[1].set_title("peri vs forward return  (5-day)")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Part 8 — 结论与下一步

## 当前框架已完成

- **数据 → panel**：3 秒 bin × UTC 日的 panel 构造（`build_intraday_panel`）；
- **三步法估计**：detrend / ACF（FFT 加速）/ inverse DFT（`estimate_spectral`）；
- **合成数据 sanity check**：在已知答案上验证恢复精度；
- **单标的频谱**：SOL/USDC 的 \(\hat a_j^2\)、fVR table、3 种度量对比；
- **跨标的对比模板**：`analyze_symbol(...)` + `cross_symbol_fVR_table` + `plot_cross_symbol_intensity`；
- **Daily peri 信号**：每日频谱 → peri → 五分位 → forward-return event study。

## BTC / ETH 拉数据后只需做的两步

```bash
# 在仓库根目录
python scripts/fetch_perpetual_trades.py \
    --symbols BTC/USDC,ETH/USDC \
    --start-date 2026-01-01
```

数据进 cache 后，回到本 notebook **从 Part 6 重新执行**——下游所有图、表、`peri` 都会自动包含三个标的。

## 第一性问题怎么回答

### Q1：BTC 是否最快？

看 Part 6 的两块产出：
- `dominant_period_summary(analyses)` 的 `period_sec` 越小 → 越"快"；
- `plot_cross_symbol_intensity` 里 *峰值 j* 越靠右 → 周期越短。

**判断标准**：把每个标的 top-3 周期的中位数作为 *characteristic period*。如果排序是 BTC < ETH < SOL，假设成立。

### Q2：peri 是否能转成币上的 alpha？

我们用 *time-series* 五分位 long-short 替代了论文的 *cross-sectional* PmS：
- 看 Q5 − Q1 在不同 forward horizon 的均值；
- 看 t-stat 是否在 1.5–2 之间；
- 同时看 OLS slope 的方向。

**警示**：单标的 \(\sim\)100 天样本下，统计显著性会很弱。要拿严肃的结论，需要：
1. 多标的（让 cross-section 能跑起来），最好扩到 ≥ 10 个 perp（`SOL/BTC/ETH/BNB/AVAX/...`）；
2. 时间扩到至少 1 年；
3. 引入因子模型（市场 beta、动量、size 用市值 proxy），看残差里的 alpha。

## 后续可加的小实验

| 实验 | 实现成本 | 预期收益 |
|---|---|---|
| 把 `n_trades` 换成 *signed n_trades* (`buy_trades − sell_trades`) | 低 | 验证论文里"signed volume 无周期性"是否在 perp 上也成立 |
| 在 *bid-ask 价差*、*volatility* 上跑同一框架 | 中 | 论文 Online Appendix F 的 crypto 版本 |
| Robinhood-outage-style 事件研究：找一次 *中心化交易所宕机* 看 peri 跳变 | 中 | 验证算法 vs 散户的角色 |
| 加入 perp funding rate 作为协变量 | 低 | 看周期性与资金成本的耦合 |
| 把 `peri` 加到 `cta/` 现有信号里做 IC + 增量 R² 检验 | 中 | 快速判断它能否给已有策略增 alpha |

## 给你的 takeaways（关于 Fourier）

1. **DFT 不是魔法，是 *投影***：把任何序列投到 \(\{\cos(\lambda_j t), \sin(\lambda_j t)\}\) 这组正交基上，每个系数告诉你"你的序列里有多少这个频率"。
2. **ACF 是这个故事的关键技巧**：raw 信号噪声大，但 ACF 在 lag ≠ 0 时把噪声"抹平"了，所以是估计周期强度的更稳定的中间量（Wiener-Khinchin）。
3. **`fVR` 比 `a²` 更可解释**：它直接告诉你"这个频率解释了多少方差"，跨标的、跨度量都可比。
4. **小心 spurious peaks**：把 baseline `1/n = 0.2%` 当作"什么都没有"的标尺；只有 fVR ≥ 1% 量级才值得当回事。